# PV combined evaluation

source (Huggingface): finloop/yolov8s-seg-solar-panels (aka Rzeszów model)

source (Huggingface): https://huggingface.co/spaces/ArielDrabkin/Solar-Panel-Detector

source (Huggingface): andrewgray11/autotrain-solar-panel-object-detection-50559120777

XI 25

*MD*

In [ ]:
from ultralytics import YOLO
from time import time

In [ ]:
datasets = {
    'pilot': "pilotPV_panels.v1i.yolov8-obb/data.yaml",
    # 'rzeszow': "rzeszowSolar panels seg.v2i.yolov8-obb/data.yaml",
    'synth': 'auto_pv_to_fine_tunning.v4i.yolov8-obb/data.yaml'
}

In [ ]:
splits = {
    'pilot': ['val'],
    'rzeszow': ['val', 'test'],
    'synth': ['train', 'val', 'test']
}

In [ ]:
models = {
    'finloop': 'best.pt',
    'ariel': 'final-mosaic-augmentation.pt',
    'pools': 'solarpanels_pools_yolov8l-p2_1024_v1.pt'
}

In [23]:
# datasets = {
#     'pilot': "pilotPV_panels.v1i.yolov8-obb/data.yaml"}
# models = {
#     'finloop': 'best.pt',
#     'ariel': 'final-mosaic-augmentation.pt'
#     }

In [ ]:
with open('evaluation_results.csv', 'a') as f:
    f.write('dataset,split,model,Class,Images,Instances,Box-P,Box-R,Box-F1,mAP50,mAP50-95,Mask-P,Mask-R,Mask-F1,t\n')

In [ ]:
for data_key, dataset in datasets.items():
    for splt in splits[data_key]:
        for model_key, model in models.items():
            bt = 16 if model_key == 'pools' else 64
            model = YOLO(model)
            suffix = ',0,0,0' if model_key != 'finloop' else ''
            t = time()
            results = model.val(data=dataset, single_cls=True, batch=bt, iou=0.7, split=splt, plots=True, project=f'runs/{data_key}_{splt}_{model_key}')
            t = time()-t
            with open('evaluation_results.csv', 'a') as f:
                f.write(f'{data_key},{splt},{model_key},{results.to_csv(decimals=3).splitlines()[1]}{suffix},{t}\n')
            print('done', model_key, data_key, splt, 'in', t)

Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8s-seg summary (fused): 85 layers, 11,779,987 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.3 ms, read: 54.5±2.0 MB/s, size: 80.7 KB)
val: Scanning /content/drive/MyDrive/ZPB/pilotPV_panels.v1i.yolov8-obb/test/labels.cache... 55 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 55/55 100.4Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.7s/it 4.7s
                   all         55        227      0.548      0.414      0.379      0.202      0.202      0.159     0.0712     0.0151
Speed: 1.2ms preprocess, 13.6ms inference, 0.0ms loss, 4.3ms postprocess per image
Results saved to /content/runs/pilot_val_finloop/val4
done finloop pilot val in 15.781958818435669
Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary